# **ADET | IOT SIMULATION DATA | GROUP 8**

**Data Constraints:**
*   CO2 (ppm): 400–5000 (ambient outdoor air usually 400–500, but can spike indoors or near sources)
*   PM2.5 (µg/m³): 0–250 (WHO guideline is <25, but polluted areas can exceed 150+)
*   PM10 (µg/m³): 0–300 (similar reasoning as PM2.5, but larger particles)
*   O3 (ppb): 0–200 (urban ozone levels typically <100, but can spike)
*   NO2 (ppb): 0–150 (urban traffic areas can reach >100)
*   Temperature (°C): 20–40 (Philippines tropical climate range)
*   Humidity (%): 40–100
*   Soil Moisture (%): 5–60 (dry to wet soil)
*   pH (water): 6.0–8.5 (typical safe range)
*   Turbidity (NTU): 0–100 (clear to very turbid water)

## **Notes on Script Design Considerations**
### 1. Geographic Distribution
  *   We needed sensors spread across Luzon, Visayas, and Mindanao for balanced representation.
  *   Chose 4 provinces per region with approximate centroid coordinates.
  *   Added a small random offset (±0.1°) to latitude/longitude so sensors aren’t all clustered exactly at the centroid, but remain realistically within the province.
  *   Ensured each sensor keeps a consistent location across all days.

### 2. Balanced Sensor Assignment
*   Total sensors: 20.
*   Distributed evenly: 7 in Luzon, 7 in Visayas, 6 in Mindanao.
*   Each sensor is tied to one province for the entire simulation.

### 3. Time Series Simulation
*   Simulation period: 30 days.
*   Daily interval starting March 1, 2026 → March 30, 2026.
*   Each sensor generates one record per day, resulting in 600 records total.

### 4. Environmental Variables & Realistic Ranges
*   See Data Constraints above

### 5. AQI Calculation
*   Used the python-aqi library (EPA standard).
*   Computed AQI from PM2.5, PM10, O₃ (8h), NO₂ (1h).
*   Added fallback logic (average pollutants) if AQI computation fails.
*   Categorized AQI into levels: Good, Moderate, Unhealthy for Sensitive Groups, Unhealthy, Very Unhealthy, Hazardous.

### 6. Consistency vs. Randomness
*   Consistent: Sensor ID, province, lat/lon remain fixed across all days.
*   Randomized daily: Environmental readings (air, climate, soil, water) vary each day to simulate real monitoring data.

### 7. Output & Testing
*   Generated a DataFrame with all records.
*   Printed sample rows for quick inspection.
*   Total records check ensures expected 600 entries.







In [ ]:
pip install python-aqi

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import aqi

# Province centroids grouped by region
province_coords = {
    # Luzon
    "Metro Manila": (14.5995, 120.9842),
    "Batangas": (13.7565, 121.0583),
    "Pampanga": (15.0794, 120.6190),
    "Bulacan": (14.7942, 120.8799),

    # Visayas
    "Cebu": (10.3157, 123.8854),
    "Iloilo": (10.7202, 122.5621),
    "Leyte": (10.9957, 124.8309),
    "Negros Occidental": (10.6750, 122.9500),

    # Mindanao
    "Davao del Sur": (7.1907, 125.4553),
    "Zamboanga del Sur": (7.8257, 123.4370),
    "Misamis Oriental": (8.5000, 124.6500),
    "Cotabato": (7.2169, 124.2486)
}

def generate_sensor_id(num):
    return f"SN-{num:03d}"

def get_aqi_level(aqi_value):
    if aqi_value <= 50:
        return "Good"
    elif aqi_value <= 100:
        return "Moderate"
    elif aqi_value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi_value <= 200:
        return "Unhealthy"
    elif aqi_value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

def soil_category(value):
    if value < 20:
        return "Dry"
    elif value <= 40:
        return "Optimal"
    else:
        return "Wet"

def random_near_province(province):
    lat, lon = province_coords[province]
    # Add small random offset (~±0.1 degrees)
    return lat + random.uniform(-0.1, 0.1), lon + random.uniform(-0.1, 0.1)

# -----------------------------
# Simulation Parameters
# -----------------------------
num_days = 30
num_sensors = 20
start_date = datetime(2026, 3, 1)

records = []

# -----------------------------
# Balanced sensor assignment
# -----------------------------
regions = {
    "Luzon": ["Metro Manila", "Batangas", "Pampanga", "Bulacan"],
    "Visayas": ["Cebu", "Iloilo", "Leyte", "Negros Occidental"],
    "Mindanao": ["Davao del Sur", "Zamboanga del Sur", "Misamis Oriental", "Cotabato"]
}

# Distribute sensors: 7 Luzon, 7 Visayas, 6 Mindanao
distribution = {"Luzon": 7, "Visayas": 7, "Mindanao": 6}

sensor_locations = {}
sensor_num = 1
for region, provinces in regions.items():
    for _ in range(distribution[region]):
        province = random.choice(provinces)
        lat, lon = random_near_province(province)
        sensor_locations[generate_sensor_id(sensor_num)] = (province, lat, lon)
        sensor_num += 1

# -----------------------------
# Data Simulation
# -----------------------------
for day in range(num_days):
    date = start_date + timedelta(days=day)

    for sensor_id, (province, lat, lon) in sensor_locations.items():

        # Air Data
        co2 = np.random.uniform(400, 5000)
        pm25 = np.random.uniform(0, 250)
        pm10 = np.random.uniform(0, 300)
        o3 = np.random.uniform(0, 200)
        no2 = np.random.uniform(0, 150)

        # Compute AQI using python-aqi (EPA standard)
        try:
            aqi_value = aqi.to_aqi([
                (aqi.POLLUTANT_PM25, str(pm25)),
                (aqi.POLLUTANT_PM10, str(pm10)),
                (aqi.POLLUTANT_O3_8H, str(o3)),
                (aqi.POLLUTANT_NO2_1H, str(no2))
            ])
        except Exception:
            aqi_value = int(np.mean([pm25, pm10, o3, no2]))  # fallback

        aqi_level = get_aqi_level(aqi_value)

        # Climate Data
        temp = np.random.uniform(20, 40)
        humidity = np.random.uniform(40, 100)

        # Soil Data
        soil_moisture = np.random.uniform(5, 60)
        soil_moisture_cat = soil_category(soil_moisture)

        # Water Data
        ph = np.random.uniform(6.0, 8.5)
        turbidity = np.random.uniform(0, 100)

        records.append({
            "Date": date.strftime("%Y-%m-%d"),
            "Sensor ID": sensor_id,
            "Location": province,
            "Latitude": round(lat, 5),
            "Longitude": round(lon, 5),

            # Air
            "CO2": round(co2, 2),
            "PM2.5": round(pm25, 2),
            "PM10": round(pm10, 2),
            "O3": round(o3, 2),
            "NO2": round(no2, 2),
            "AQI": aqi_value,
            "AQI Level": aqi_level,

            # Climate
            "Temperature": round(temp, 2),
            "Humidity": round(humidity, 2),

            # Soil
            "Soil Moisture": round(soil_moisture, 2),
            "Soil Moisture Category": soil_moisture_cat,

            # Water
            "pH": round(ph, 2),
            "Turbidity": round(turbidity, 2)
        })

# -----------------------------
# Create DataFrame & Export
# -----------------------------
df = pd.DataFrame(records)

# For testing: show first 35 rows
print(df.head(35))
print(f"\nTotal records generated: {len(df)}")

df.to_csv("iot_data.csv", index=False)

          Date Sensor ID           Location  Latitude  Longitude      CO2  \
0   2026-03-01    SN-001           Batangas  13.77241  121.02809  1422.75   
1   2026-03-01    SN-002       Metro Manila  14.56538  120.92161  2823.25   
2   2026-03-01    SN-003           Pampanga  15.17610  120.59747  2837.28   
3   2026-03-01    SN-004            Bulacan  14.84333  120.84052  3284.93   
4   2026-03-01    SN-005           Pampanga  15.16884  120.55406  2428.24   
5   2026-03-01    SN-006           Pampanga  15.15867  120.53867  2793.75   
6   2026-03-01    SN-007           Pampanga  15.16475  120.66711  2563.10   
7   2026-03-01    SN-008              Leyte  11.04576  124.75037  3904.92   
8   2026-03-01    SN-009              Leyte  10.93680  124.89435  2087.87   
9   2026-03-01    SN-010               Cebu  10.28412  123.80402  1319.57   
10  2026-03-01    SN-011             Iloilo  10.63519  122.61273  3969.19   
11  2026-03-01    SN-012               Cebu  10.34147  123.88109   839.61   